# Part 1: Checking for quality issues

For this first part I want to check if data is missing, broken or suspicious. Below is a list of questions that this part intends to answer.

* What data do we have that shouldn't be null?
user_id, name, time, rating, gmap_id

* What metadata types do we have that shouldn't be null?
name, address, gmap_id, category, avg_rating, num_of_reviews

* Do we have duplicate entries in our data or metadata?

In [ ]:
import sys
import pandas as pd


# open the JSON files
try:
    rd1 = pd.read_json('data/review-Georgia.json', lines=True, chunksize=1000)
    rd2 = pd.read_json('data/meta-Georgia.json', lines=True, chunksize=1000)
except FileNotFoundError:
    print("File not found")
    sys.exit()

# Check for corresponding gmap_ids
# set up sets because we don't need to worry about duplicates... yet
gmap_id_dset = set()
gmap_id_mset = set()
# append set in chunks for both data and metadata
for chunk in rd1:
    gmap_id_dset|=set(chunk['gmap_id'])
for chunk in rd2:
    gmap_id_mset|=set(chunk['gmap_id'])
# check for gmap_id in data with no corresponding gmap_id in metadata
missing_list = gmap_id_dset - gmap_id_mset
if len(missing_list) > 0:
    print("Missing gmap_ids found:")
    print(list(missing_list))
else:
    print("There are no missing gmap_ids")



It is reassuring to know that there are no missing gmap_ids.

Now to check for duplicates. I decided to count any part of the metadata with the same Name, Address, and gmap_id as a duplicate since no two restaurants would share these three variables and not be a duplicate.

In [3]:
import sys
import pandas as pd

# open the JSON files
try:
    rd2 = pd.read_json('data/meta-Georgia.json', lines=True)
except FileNotFoundError:
    print("File not found")
    sys.exit()

print("Number of duplicated rows in metadata:")
print(rd2.duplicated(subset=['name', 'address', 'gmap_id']).sum())


Number of duplicated rows in metadata:
986


A bit concerning. We'll have to remove this after we get the number of duplicated rows in the data.

This will have to be done slightly differently as the file in question is too large for my computer to have the whole file open at once. I will open it in chunks.

In [ ]:
import sys
import pandas as pd

duplicate_count = 0
try:
    keys_seen = set()
    data = pd.read_json('data/review-Georgia.json', lines=True, chunksize=1000)
    for chunk in data:
        keys = zip(chunk['user_id'], chunk['time'], chunk['gmap_id'])
        for key in keys:
            if key in keys_seen:
                duplicate_count += 1
            else:
                keys_seen.add(key)
except FileNotFoundError:
    print("File not found")
    sys.exit()

print("Number of duplicated rows in metadata:")
print(duplicate_count)


Tens of thousands of duplicates is still fairly terrible. Now to remove these duplicates.

# Part 2: Complete transformation

The objectives of these two scripts are to:
* Remove duplicates and useless labels
* Replace the pics with the number of pictures
* Place the new .json file within the data folder


In [11]:
import pandas as pd
try:
    rd1 = pd.read_json('data/meta-Georgia.json', lines=True)
    rd1.drop(labels=['description', 'price', 'MISC', 'state', 'relative_results', 'url'], axis=1, inplace=True)
    # drop duplicates from file
    rd1.drop_duplicates(subset=['name', 'address', 'gmap_id'], inplace=True)
    # return cleaned file
    rd1.to_json('data/meta-Georgia_cleaned.json', orient='records', lines=True)
except FileNotFoundError:
    print("File not found")

This reduced the size of the metadata file by 74.39%

In [13]:
import pandas as pd
keys_seen = set()
boolean_mask = []
try:
    data = pd.read_json('data/review-Georgia.json', lines=True, chunksize=1000, dtype={'user_id':str, 'gmap_id':str})
    for chunk in data:
        chunk.drop(labels=['name', 'text', 'resp'], axis=1, inplace=True)
        chunk['pics'] = chunk['pics'].apply(lambda x: len(x) if isinstance(x, list) else 0)
        keys = chunk[['user_id', 'time', 'gmap_id']].itertuples(index=False, name=None)
        for key in keys:
            if key in keys_seen:
                boolean_mask.append(False)
            else:
                boolean_mask.append(True)
                keys_seen.add(key)
        chunk[boolean_mask].to_json('data/review-Georgia_cleaned.json', orient='records', lines=True, mode='a')
        boolean_mask = []
except FileNotFoundError:
    print("File not found")
except Exception as e:
    print(e)


This reduced the size of the data file by 61.21%